In [39]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import sequence

In [40]:
#Load the IMDB dataset
word_index = imdb.get_word_index()
reversed_word_index = {value:key for key,value in word_index.items()}
reversed_word_index[12345]

'liberated'

In [41]:
#Load the pre trained model

model = load_model('rnn_model.h5')
model.summary()
#model.get_weights()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (32, 200, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [42]:
import re
#Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reversed_word_index.get(i-3,'?') for i in encoded_review])

#Function to preprocess user review
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()

    encoded_review = []

    for word in words:
        index = word_index.get(word)

        if index is not None and index < 10000:
            encoded_review.append(index + 3)
        else:
            encoded_review.append(2)

    padded_review = sequence.pad_sequences(
        [encoded_review],
        maxlen=200
    )

    return padded_review

In [43]:
#Prediction Function
def predict_sentiment(review):
    preprocessed_text = preprocess_text(review)

    prediction = model.predict(preprocessed_text)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

In [45]:
#Sample input and prediction
review = "This is one of the greatest movies i have ever seen"
print(predict_sentiment(review))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
('Negative', np.float32(0.38960317))


In [46]:
reviews = [
    "This movie was absolutely fantastic",
    "I loved this film",
    "Worst movie ever",
    "Terrible acting and poor story",
    "Amazing screenplay and acting"
]

for r in reviews:
    print(r)
    print(predict_sentiment(r))
    print()

This movie was absolutely fantastic
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
('Negative', np.float32(0.082755))

I loved this film
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
('Negative', np.float32(0.24316329))

Worst movie ever
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
('Negative', np.float32(0.03295369))

Terrible acting and poor story
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
('Negative', np.float32(0.06047954))

Amazing screenplay and acting
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
('Negative', np.float32(0.17881253))

